# Object, intensity and radial features

> **Notebook role:** production-style construction of interpretable 2D/3D nuclear and spheroid measurements.

## 1. Build the object table

Every labelled nucleus receives a stable label, pixel-based size, centroid,
bounding box and intensity summary. Two-dimensional objects additionally expose
eccentricity, axis lengths and solidity.

In [ ]:
from pathlib import Path
import numpy as np
from tifffile import imread

from nuclear_imaging_core.measurements import region_feature_table

segmentation = np.load(Path("outputs/segmentation.npz"))
image = segmentation["image"]
labels = segmentation["labels"]
foreground = segmentation["foreground"]
nuclear_features = region_feature_table(image, labels)
nuclear_features.insert(0, "object_id", nuclear_features["label"].map(lambda value: f"nucleus-{value}"))
nuclear_features.head()

## 2. Add texture, curvature and intensity-distribution descriptors

Dedicated 2D and 3D namespaces keep dimension-dependent implementations
explicit while presenting the same method families.

In [ ]:
from nuclear_imaging_core.features import two_d, three_d

feature_families = {
    "2D": [two_d.shapeFeatures, two_d.textureFeatures, two_d.curvatureFeatures],
    "3D": [three_d.shapeFeatures, three_d.textureFeatures, three_d.curvatureFeatures],
}
{dimension: [function.__name__ for function in functions] for dimension, functions in feature_families.items()}

## 3. Measure radial organization

Normalized shells describe centre-to-boundary intensity organization within a
nucleus. Spheroid analysis uses configured pixel distances for inward and
outward nuclear distributions.

In [ ]:
from nuclear_imaging_core.measurements import normalized_radial_profile
from nuclear_spheroid_analysis import measure_spheroid_system
from microscopy_workflows import load_config

radial_config = load_config("spheroid_radial.json")
spheroid_labels = imread(Path("data/labels/spheroids.tif"))
profile = normalized_radial_profile(image, foreground, bins=10)
nuclei, spheroids = measure_spheroid_system(image, labels, spheroid_labels)
profile.head(), nuclei.head(), spheroids.head(), radial_config

## 4. Save identity-preserving feature tables

CSV outputs retain object identifiers and pixel-based measurements for graph,
embedding and statistical stages.

In [ ]:
output_directory = Path("outputs/features")
output_directory.mkdir(parents=True, exist_ok=True)
nuclear_features.to_csv(output_directory / "nuclei.csv", index=False)
nuclei.to_csv(output_directory / "nuclei_with_spheroids.csv", index=False)
spheroids.to_csv(output_directory / "spheroids.csv", index=False)

## Representative result

Inspect each nuclear crop beside its object mask and within-mask intensity
partition before reducing those pixels to morphology, texture and radial values.

![Nuclear masks and intensity partitions](../assets/results/nucleus-intensity-partition.png)